# Extract Email-id

In [1]:
import pandas as pd

In [2]:
import re

In [3]:
df = pd.read_csv("../data/processed/cleaned_resume.csv")

print(df.shape)
print(df.columns)

(2481, 5)
Index(['ID', 'resume_text', 'Category', 'clean_resume', 'text_length'], dtype='str')


In [4]:
print(df["resume_text"].iloc[0])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [5]:
def extract_email(text):
    pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
    
    match = re.search(pattern, text)
    
    if match:
        return match.group()
    
    return None

In [6]:
test_text = "Contact me at akash@example.com"

print(extract_email(test_text))

akash@example.com


In [7]:
df["email"] = df["resume_text"].apply(extract_email)

In [8]:
df[["ID", "email"]].head(10)

,ID,email
0,16852973,NaN
1,22323967,NaN
2,33176873,NaN
3,27018550,NaN
4,17812897,NaN
5,11592605,NaN
6,25824789,NaN
7,15375009,NaN
8,11847784,NaN
9,32896934,NaN


In [9]:
print("Emails found:", df["email"].notna().sum())

Emails found: 19


In [10]:
df[df["email"].notna()][["ID", "email"]].head(20)

,ID,email
258,90867631,william.cook2@conus.army.mil
386,23398843,Lamaestram4kidscare@yahoo.com
921,30437583,Cheryl.Lewis@fns.usda.gov
926,38216888,roger.thilmony@ars.usda.gov
948,17640785,elisabeth.hsu@anthro.ox.ac.uk
958,26835781,ameredith@desu.edu
1171,95429627,Greg@TheVarium.com
1201,22571461,Bradybiggs918@gmail.com
1220,26919036,Novella.Walton@VerizonWireless.com
1574,59818742,patelsujaym54@yahoo.com


In [11]:
row = df[df["email"].notna()].iloc[0]

print(row["email"])
print(row["resume_text"][:1000])

william.cook2@conus.army.mil
         INFORMATION TECHNOLOGY SPECIALIST           Career Overview    Permanent            Certifications    ITNW 1458 CompTIA Network+ ITMT 1350 MS 70-270 Managing and Maintaining an MS Windows 2003 Environment ITMT 1355 Server 2003 Network Infrastructure ITMT 1400 MS 70-270 Implementing and Supporting MS Windows XP Professional Job Related   Microsoft Certified IT Professional (MCITP) Apr 2012 Training:     CompTIA Security+ Certified Professional (Mar 2010) Microsoft Certified Professional (MCP) Oct 2008 CompTIA Network+ Certified Professional (May 2008) CompTIA A+ Certified Professional (Apr 2008) Language      Language          Spoken          Written          Read Skills:       Spanish          Advanced          Advanced          Advanced References:   Name        Employer          Title        Phone        Email William     407th AFSB        Supervisor   254-287-7798 william.cook2@conus.army.mil Cook (*) Mr. John    First Army        Chief,      25

In [12]:
print("Total emails:", df["email"].notna().sum())
print("Unique emails:", df["email"].nunique())

Total emails: 19
Unique emails: 19


# Extract Phone Number

In [13]:
def extract_phone(text):
    pattern = r'(?<!\d)(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}(?!\d)'
    
    match = re.search(pattern, text)
    
    if match:
        return match.group()
    
    return None

In [14]:
test_text = "Contact: 254-287-7798"

print(extract_phone(test_text))

254-287-7798


In [15]:
df["phone"] = df["resume_text"].apply(extract_phone)

In [16]:
df[["ID", "phone"]].head(20)

,ID,phone
0,16852973,NaN
1,22323967,NaN
2,33176873,NaN
3,27018550,NaN
4,17812897,NaN
5,11592605,NaN
6,25824789,NaN
7,15375009,NaN
8,11847784,NaN
9,32896934,NaN


In [17]:
print("Phone numbers found:", df["phone"].notna().sum())

Phone numbers found: 77


In [18]:
df[df["phone"].notna()][["ID", "phone"]].head(20)

,ID,phone
81,24508725,(217) 097-5477
94,20806155,(910-432-2392
218,21780877,865) 336-4800
226,12045067,(520) 459-3174
258,90867631,254-287-7798
321,27058381,304-264-5413
346,33704389,318-243-9294
353,48547319,502-868-6764
370,34465087,770-992-4006
371,24791126,2009000200007


In [19]:
row = df[df["phone"].notna()].iloc[0]

print("Extracted phone:", row["phone"])
print()
print(row["resume_text"][:1500])

Extracted phone: (217) 097-5477

         FIELD HR ASSOCIATE           Summary    Reliable HR Field Associate with a Master's of science in Human Resource management emphasis as a Generalist. Passionate and motivated with a drive for excellence.  Handles tasks with accuracy and efficiency. 
______________________________________________________________________________
Skills
*Verbal and written communication skills 
*Time management and organization skills
*Powerpoint presentations
*Flexible 
*Recruiting
*Investigation training
*Employee engagement
*Ability to develop and execute recruitment strategies
*Analyze data and recommend opportunities for improvement 
*Experience in preparing for audits
*Assisted in the hiring process by screening resumes-Select interview process certified via FedEx Ground
* Demonstrated ability to interact effectively with employees. 
* Ability to anticipate and identify problems and use sound judgment and fact based analysis to develop effective and efficien

In [20]:
print("Total phone numbers:", df["phone"].notna().sum())
print("Unique phone numbers:", df["phone"].nunique())

Total phone numbers: 77
Unique phone numbers: 77


# Extract Name

In [26]:
def extract_name(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    
    if not lines:
        return None
    
    rejected_words = {
        "summary",
        "objective",
        "skills",
        "experience",
        "education",
        "profile",
        "career",
        "overview",
        "professional",
        "employment",
        "references",
        "contact",
        "certifications",
        "qualifications",
        "responsibilities",
        "achievements",
        "projects",
        "information",
        "technology",
        "associate",
        "specialist",
        "manager",
        "developer",
        "engineer"
    }
    
    for line in lines[:10]:
        words = line.split()
        
        # Name should normally contain 2–4 words
        if not 2 <= len(words) <= 4:
            continue
        
        # Reject very long lines
        if len(line) > 40:
            continue
        
        # Only alphabetic words, allowing - and '
        if not all(
            word.replace("-", "").replace("'", "").isalpha()
            for word in words
        ):
            continue
        
        lower_words = {word.lower() for word in words}
        
        # Reject common resume headings
        if lower_words & rejected_words:
            continue
        
        # Reject sentence-like lines
        if any(word.lower() in {
            "and", "with", "the", "to", "for", "of",
            "in", "on", "from", "as", "by"
        } for word in words):
            continue
        
        return line
    
    return None

In [22]:
test_text = """
John Smith
Software Engineer

Experienced software developer with Python and SQL skills.
"""

print(extract_name(test_text))

John Smith


In [23]:
df["name"] = df["resume_text"].apply(extract_name)

In [24]:
df[["ID", "name"]].head(20)

,ID,name
0,16852973,NaN
1,22323967,Implemented programs to achieve
2,33176873,Provided coaching
3,27018550,NaN
4,17812897,NaN
5,11592605,NaN
6,25824789,NaN
7,15375009,NaN
8,11847784,NaN
9,32896934,NaN


In [25]:
print("Names found:", df["name"].notna().sum())

Names found: 181


In [27]:
df["name"] = df["resume_text"].apply(extract_name)

In [28]:
df[["ID", "name"]].head(20)

,ID,name
0,16852973,NaN
1,22323967,brand awareness
2,33176873,Provided coaching
3,27018550,NaN
4,17812897,NaN
5,11592605,NaN
6,25824789,NaN
7,15375009,NaN
8,11847784,NaN
9,32896934,NaN


In [29]:
print("Names found:", df["name"].notna().sum())

Names found: 101


In [30]:
import spacy

In [31]:
print(spacy.__version__)

3.8.16


In [32]:
nlp = spacy.load("en_core_web_sm")

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("../data/processed/cleaned_resume.csv")

print(df.shape)

(2481, 5)


In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully!")

spaCy model loaded successfully!


In [4]:
text = df["resume_text"].iloc[258]

doc = nlp(text)

for ent in doc.ents:
    if ent.label_ == "PERSON":
        print(ent.text)

Email William
Cook
John    
Maintained
Visio
Maintained MS
Supervisor
John Ciesiolka
Maintained
Performed
Established
Daily
Maintained
Supervisor
William Cook
Training
Supervisor
Paul Mercator
Relevant Coursework
Licenses
Active Directory
C++
Email
MS Windows
Oct
Spreadsheet
Visio


In [5]:
text = df["resume_text"].iloc[258]

print(text[:1500])

         INFORMATION TECHNOLOGY SPECIALIST           Career Overview    Permanent            Certifications    ITNW 1458 CompTIA Network+ ITMT 1350 MS 70-270 Managing and Maintaining an MS Windows 2003 Environment ITMT 1355 Server 2003 Network Infrastructure ITMT 1400 MS 70-270 Implementing and Supporting MS Windows XP Professional Job Related   Microsoft Certified IT Professional (MCITP) Apr 2012 Training:     CompTIA Security+ Certified Professional (Mar 2010) Microsoft Certified Professional (MCP) Oct 2008 CompTIA Network+ Certified Professional (May 2008) CompTIA A+ Certified Professional (Apr 2008) Language      Language          Spoken          Written          Read Skills:       Spanish          Advanced          Advanced          Advanced References:   Name        Employer          Title        Phone        Email William     407th AFSB        Supervisor   254-287-7798 william.cook2@conus.army.mil Cook (*) Mr. John    First Army        Chief,      254-553-5006 Ciesiolka   Divisi

In [6]:
test_indices = [0, 100, 500, 1000, 1500, 2000]

for i in test_indices:
    print("\n" + "=" * 60)
    print("Resume ID:", df.iloc[i]["ID"])
    print(df.iloc[i]["resume_text"][:500])


Resume ID: 16852973
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-task

Resume ID: 10694288
         HR BENEFITS/LEAVE COORDINATOR       Summary    13 years of Human Resources experience and 27 years of administrative experience working in various settings
*Professional, detail-oriented, excellent time-management skills, team player, open and honest communicator, effective at maintaining confidentiality and multi-tasker.      Skills        Proficient in all Microsoft applications and HRIS systems (Banner, PeopleAdmin, Personnel Management Info

In [7]:
test_indices = [0, 100, 500, 1000, 1500, 2000]

for i in test_indices:
    text = df.iloc[i]["resume_text"]
    doc = nlp(text[:300])

    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

    print("\nResume ID:", df.iloc[i]["ID"])
    print("PERSON entities in first 300 chars:", persons)


Resume ID: 16852973
PERSON entities in first 300 chars: []

Resume ID: 10694288
PERSON entities in first 300 chars: []

Resume ID: 21614256
PERSON entities in first 300 chars: []

Resume ID: 29134721
PERSON entities in first 300 chars: []

Resume ID: 18975686
PERSON entities in first 300 chars: []

Resume ID: 69764348
PERSON entities in first 300 chars: []


In [8]:
df["name"] = None

In [9]:
df[["ID", "name", "email", "phone"]].head(10)

KeyError: "['email', 'phone'] not in index"

In [ ]:
import pandas as pd
import re

df = pd.read_csv("../data/processed/cleaned_resume.csv")

print(df.shape)

In [ ]:
def extract_email(text):
    pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
    
    match = re.search(pattern, text)
    
    if match:
        return match.group()
    
    return None

In [ ]:
df["email"] = df["resume_text"].apply(extract_email)

In [ ]:
def extract_phone(text):
    pattern = r'(?<!\d)(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}(?!\d)'
    
    match = re.search(pattern, text)
    
    if match:
        return match.group()
    
    return None

In [ ]:
df["phone"] = df["resume_text"].apply(extract_phone)

In [ ]:
df["name"] = None

In [ ]:
df[["ID", "name", "email", "phone"]].head(10)

In [19]:
print(df[["ID", "name", "email", "phone"]].head(10).to_string())

         ID  name email phone
0  16852973  None   NaN   NaN
1  22323967  None   NaN   NaN
2  33176873  None   NaN   NaN
3  27018550  None   NaN   NaN
4  17812897  None   NaN   NaN
5  11592605  None   NaN   NaN
6  25824789  None   NaN   NaN
7  15375009  None   NaN   NaN
8  11847784  None   NaN   NaN
9  32896934  None   NaN   NaN


# Education Extraction.

In [30]:
education_patterns = [
    r'\bB\.Tech\b',
    r'\bB\.E\.\b',
    r'\bB\.Sc\.\b',
    r'\bB\.A\.\b',
    r'\bB\.Com\.\b',
    r'\bM\.Tech\b',
    r'\bM\.E\.\b',
    r'\bM\.Sc\.\b',
    r'\bM\.A\.\b',
    r'\bM\.Com\.\b',
    r'\bMBA\b',
    r'\bMCA\b',
    r'\bPh\.D\.\b',
    r'\bBachelor of (?:Arts|Science|Engineering|Technology|Education|Commerce|Business Administration)\b',
    r'\bMaster of (?:Arts|Science|Engineering|Technology|Education|Business Administration|Public Administration|Commerce)\b'
]

In [31]:
def extract_education(text):
    education = []
    
    for pattern in education_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        
        for match in matches:
            if isinstance(match, tuple):
                match = match[0]
            
            match = match.strip()
            
            if match and match.lower() not in [x.lower() for x in education]:
                education.append(match)
    
    return education

In [32]:
df["education"] = df["resume_text"].apply(extract_education)

In [33]:
print(df[["ID", "education"]].head(20).to_string())

          ID                                                 education
0   16852973                                                        []
1   22323967                        [Bachelor of Arts, Master of Arts]
2   33176873                                     [Bachelor of Science]
3   27018550                                                        []
4   17812897                     [Bachelor of Business Administration]
5   11592605                                                        []
6   25824789                                     [Bachelor of Science]
7   15375009                                [MBA, Bachelor of Science]
8   11847784                                        [Bachelor of Arts]
9   32896934                                                        []
10  29149998                                                        []
11  11480899  [Bachelor of Education, Master of Public Administration]
12  23155093                                                        []
13  11

In [34]:
education_count = df["education"].apply(len)

print("Total resumes:", len(df))
print("Resumes with education:", (education_count > 0).sum())
print("Resumes without education:", (education_count == 0).sum())

Total resumes: 2481
Resumes with education: 1229
Resumes without education: 1252


In [35]:
print(df["education"].explode().value_counts().head(20))

education
Bachelor of Science                    642
Bachelor of Arts                       353
Master of Science                      169
MBA                                    167
Master of Business Administration       66
Master of Arts                          54
Bachelor of Business Administration     32
Bachelor of Engineering                 29
Bachelor of Commerce                    12
Bachelor of Technology                  11
BACHELOR OF SCIENCE                     10
Master of Education                      9
BACHELOR OF ARTS                         8
MCA                                      8
Master of Public Administration          5
Bachelor Of Science                      4
MASTER OF SCIENCE                        4
Bachelor of Education                    3
M.A.                                     3
Bachelor of science                      2
Name: count, dtype: int64


In [39]:
def normalize_education(education_list):
    normalized = []

    for education in education_list:
        education = education.lower().strip()

        if education == "bachelor of science":
            education = "Bachelor of Science"

        elif education == "bachelor of arts":
            education = "Bachelor of Arts"

        elif education == "master of science":
            education = "Master of Science"

        elif education == "master of arts":
            education = "Master of Arts"

        elif education == "bachelor of engineering":
            education = "Bachelor of Engineering"

        elif education == "bachelor of technology":
            education = "Bachelor of Technology"

        elif education == "bachelor of commerce":
            education = "Bachelor of Commerce"

        elif education == "bachelor of business administration":
            education = "Bachelor of Business Administration"

        elif education == "master of business administration":
            education = "Master of Business Administration"

        elif education == "master of education":
            education = "Master of Education"

        elif education == "bachelor of education":
            education = "Bachelor of Education"

        elif education == "master of public administration":
            education = "Master of Public Administration"

        elif education == "mba":
            education = "MBA"

        elif education == "mca":
            education = "MCA"

        elif education == "m.a.":
            education = "M.A."

        elif education == "master of commerce":
            education = "Master of Commerce"
        
        elif education == "m.e.":
            education = "M.E."
        
        elif education == "b.a.":
            education = "B.A."
        
        elif education == "master of engineering":
            education = "Master of Engineering"
        
        elif education == "master of technology":
            education = "Master of Technology"

        normalized.append(education)

    return list(dict.fromkeys(normalized))

In [43]:
df["education"] = df["education"].apply(normalize_education)

In [44]:
print(df["education"].explode().value_counts().head(20))

education
Bachelor of Science                    658
Bachelor of Arts                       362
Master of Science                      174
MBA                                    167
Master of Business Administration       68
Master of Arts                          54
Bachelor of Business Administration     32
Bachelor of Engineering                 29
Bachelor of Commerce                    14
Bachelor of Technology                  12
Master of Education                      9
MCA                                      8
Master of Public Administration          5
Bachelor of Education                    3
M.A.                                     3
Master of Commerce                       2
M.E.                                     1
B.A.                                     1
Master of Engineering                    1
Master of Technology                     1
Name: count, dtype: int64


In [45]:
df.to_csv("../data/processed/resume_information_extracted.csv", index=False)

In [46]:
print(df.shape)
print(df.columns.tolist())

(2481, 9)
['ID', 'resume_text', 'Category', 'clean_resume', 'text_length', 'email', 'phone', 'name', 'education']


# Experience Extraction

In [47]:
experience_patterns = [
    r'(\d+(?:\.\d+)?)\+?\s+years?\s+of\s+experience',
    r'(\d+(?:\.\d+)?)\+?\s+years?\s+experience',
    r'(\d+(?:\.\d+)?)\+?\s+years?\s+in'
]

In [48]:
def extract_experience(text):
    experiences = []

    for pattern in experience_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        for match in matches:
            try:
                experiences.append(float(match))
            except ValueError:
                pass

    if experiences:
        return max(experiences)

    return 0.0

In [49]:
df["experience"] = df["resume_text"].apply(extract_experience)

In [50]:
print(df[["ID", "experience"]].head(20).to_string())

          ID  experience
0   16852973        15.0
1   22323967         0.0
2   33176873        20.0
3   27018550         0.0
4   17812897         0.0
5   11592605         0.0
6   25824789         0.0
7   15375009         0.0
8   11847784        15.0
9   32896934         0.0
10  29149998         0.0
11  11480899         0.0
12  23155093         0.0
13  11763983         0.0
14  27490876         0.0
15  32977530         0.0
16  93002334         0.0
17  24184357         0.0
18  73077810         0.0
19  13879043         7.0


In [51]:
print("Resumes with detected experience:", (df["experience"] > 0).sum())
print("Resumes without detected experience:", (df["experience"] == 0).sum())

print("\nExperience statistics:")
print(df["experience"].describe())

Resumes with detected experience: 387
Resumes without detected experience: 2094

Experience statistics:
count    2481.000000
mean        1.963926
std         5.511682
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        38.000000
Name: experience, dtype: float64


In [52]:
print("\nMost common experience values:")
print(df["experience"].value_counts().sort_index().head(30))


Most common experience values:
experience
0.0     2094
1.0        1
2.0       15
3.0       23
3.5        1
4.0       17
5.0       25
5.5        1
6.0       18
6.5        1
7.0       18
8.0       17
8.5        1
9.0        9
10.0      50
11.0       5
12.0      16
13.0      11
14.0       8
15.0      41
16.0      10
17.0       1
18.0       5
19.0       1
20.0      43
21.0       4
22.0       5
23.0       1
24.0       1
25.0      10
Name: count, dtype: int64


In [60]:
from datetime import datetime

def extract_date_ranges(text):
    patterns = [
        # 03/2016 to 08/2017
        r'(\d{1,2})/(\d{4})\s+to\s+(current|\d{1,2}/\d{4})',

        # Dec 2013 to Current
        r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{4})\s+to\s+(current|(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4})',

        # January 2013 to January 2015
        r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})\s+to\s+(current|(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4})'
    ]

    matches = []

    for pattern in patterns:
        matches.extend(re.findall(pattern, text, flags=re.IGNORECASE))

    return matches

In [61]:
from datetime import datetime
import re

def parse_date(month, year):
    return datetime(int(year), int(month), 1)


def month_number(month):
    month = month.lower()

    months = {
        "jan": 1,
        "january": 1,
        "feb": 2,
        "february": 2,
        "mar": 3,
        "march": 3,
        "apr": 4,
        "april": 4,
        "may": 5,
        "jun": 6,
        "june": 6,
        "jul": 7,
        "july": 7,
        "aug": 8,
        "august": 8,
        "sep": 9,
        "september": 9,
        "oct": 10,
        "october": 10,
        "nov": 11,
        "november": 11,
        "dec": 12,
        "december": 12
    }

    return months.get(month)


def calculate_date_experience(text):

    patterns = [

        # 03/2016 to 08/2017
        r'(\d{1,2})/(\d{4})\s+to\s+(current|\d{1,2}/\d{4})',

        # Dec 2013 to Current
        r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{4})\s+to\s+(current|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4})',

        # January 2013 to January 2015
        r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})\s+to\s+(current|(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4})'
    ]

    periods = []
    current_date = datetime.now()

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:

            start_month = match[0]
            start_year = match[1]
            end_value = match[2]

            try:

                # Start date
                if start_month.isdigit():
                    start_date = parse_date(
                        start_month,
                        start_year
                    )
                else:
                    month = month_number(start_month)

                    if month is None:
                        continue

                    start_date = parse_date(
                        month,
                        start_year
                    )

                # End date
                if end_value.lower() == "current":

                    end_date = current_date

                else:

                    if "/" in end_value:

                        end_month, end_year = end_value.split("/")

                        end_date = parse_date(
                            end_month,
                            end_year
                        )

                    else:

                        parts = end_value.split()

                        end_month = month_number(parts[0])
                        end_year = parts[1]

                        end_date = parse_date(
                            end_month,
                            end_year
                        )

                if end_date >= start_date:
                    periods.append(
                        (start_date, end_date)
                    )

            except (ValueError, IndexError):
                continue

    if not periods:
        return 0.0

    # Sort periods
    periods.sort(key=lambda x: x[0])

    # Merge overlapping periods
    merged_periods = [periods[0]]

    for start, end in periods[1:]:

        last_start, last_end = merged_periods[-1]

        if start <= last_end:

            if end > last_end:
                merged_periods[-1] = (
                    last_start,
                    end
                )

        else:

            merged_periods.append(
                (start, end)
            )

    # Calculate total months
    total_months = 0

    for start, end in merged_periods:

        months = (
            (end.year - start.year) * 12
            + (end.month - start.month)
        )

        total_months += months

    return round(total_months / 12, 1)

In [62]:
df["date_experience"] = df["resume_text"].apply(
    calculate_date_experience
)

In [63]:
print(
    df.loc[
        [0, 8, 18],
        ["ID", "experience", "date_experience"]
    ].to_string()
)

          ID  experience  date_experience
0   16852973        15.0             25.8
8   11847784        15.0             19.0
18  73077810         0.0              9.0


In [57]:
print(df.loc[0, "resume_text"][:3000])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [58]:
print(df.loc[8, "resume_text"][:3000])

         HR SPECIALIST       Summary    Possess 15+ years of experience as an HR Classification/Compensation Lead and as an HR Specialist, Recruitment. Managed recruitment efforts that included qualifying candidates based on documented knowledge, skills, and abilities to match organizational resource needs. Ensured candidate fulfillment of the preemployment process; i.e., background checks, drug screens, and reference checks. Developed and implemented competency and skill-based pay programs for a variety of occupations including medical (nurses), mechanical (machine operators), and civil engineers. Led a statewide review of jobs (18,000 employees) toward determining appropriate FLSA designations (e.g., executive, administrative, computer-related, creative). Provided executive and middle management consultation toward acquiring class/levels and roles necessary to accomplish departmental goals and objectives that resulted in mission successes. Training and Development: Eight years of exp

In [59]:
print(df.loc[18, "resume_text"][:3000])

         HR GENERALIST/RECRUITER           Summary     Human Resource Generalist who is an great communicator and negotiator with strong conflict resolution skills. Extensive background in recruitment and retention, mediation and records management.       Highlights         ADP, People fluent, Microsoft Offices, Interviewing expertise  Performance management strategies  Benefits administrator  Employee relations  Benefits administrator  Manager coaching and training  HRIS applications proficient  Payroll expertise  National Human Resources Association  Event management  Employee handbook development  QuickBooks proficiency  Alternative Dispute Resolution (ADR)  Staffing and recruiting professional              Accomplishments     Led the Staffing Planning Committee for [Number] years.Introduced the first passive Open Enrollment process.Human Resources   Reduced employee turnover by 10%.         Experience      HR Generalist/Recruiter    January 2013   to   January 2015     Company Name

In [64]:
df["final_experience"] = df.apply(
    lambda row: row["experience"]
    if row["experience"] > 0
    else row["date_experience"],
    axis=1
)

In [65]:
print(
    df.loc[
        [0, 8, 18],
        ["ID", "experience", "date_experience", "final_experience"]
    ].to_string()
)

          ID  experience  date_experience  final_experience
0   16852973        15.0             25.8              15.0
8   11847784        15.0             19.0              15.0
18  73077810         0.0              9.0               9.0


In [66]:
print("Experience detected:", (df["final_experience"] > 0).sum())
print("No experience detected:", (df["final_experience"] == 0).sum())

print("\nExperience statistics:")
print(df["final_experience"].describe())

Experience detected: 2351
No experience detected: 130

Experience statistics:
count    2481.000000
mean       16.587142
std        10.054986
min         0.000000
25%         9.000000
50%        16.200000
75%        23.100000
max        52.700000
Name: final_experience, dtype: float64


In [67]:
print("\nMost common experience values:")
print(
    df["final_experience"]
    .value_counts()
    .sort_index()
    .head(30)
)


Most common experience values:
final_experience
0.0    130
0.2      1
0.4      1
0.6      2
0.7      3
0.8      8
0.9      3
1.0      3
1.1      2
1.2      3
1.3      1
1.4      4
1.5      4
1.6      5
1.7      4
1.8      5
1.9      5
2.0     18
2.1      4
2.2     10
2.3      5
2.4      4
2.5      2
2.6      2
2.7      4
2.8      5
2.9      2
3.0     31
3.1      3
3.2      4
Name: count, dtype: int64


In [68]:
print("\nHighest experience values:")
print(
    df[
        ["ID", "experience", "date_experience", "final_experience"]
    ]
    .sort_values("final_experience", ascending=False)
    .head(15)
    .to_string(index=False)
)


Highest experience values:
      ID  experience  date_experience  final_experience
78107631         0.0             52.7              52.7
19072267         0.0             52.5              52.5
24412546         0.0             51.7              51.7
24709432         0.0             48.2              48.2
13405229         0.0             47.5              47.5
12654876         0.0             47.1              47.1
31362807         0.0             47.0              47.0
12341902         0.0             46.3              46.3
96260484         0.0             46.1              46.1
15790602         0.0             45.3              45.3
33527446         0.0             44.7              44.7
18365443         0.0             43.7              43.7
94417768         0.0             43.7              43.7
16280971         0.0             43.2              43.2
21511817         0.0             43.1              43.1


In [69]:
df["final_experience"] = df["experience"]

In [70]:
print("Experience detected:", (df["final_experience"] > 0).sum())
print("No experience detected:", (df["final_experience"] == 0).sum())

Experience detected: 387
No experience detected: 2094


In [71]:
df["final_experience"] = df["experience"]

In [72]:
df.to_csv(
    "../data/processed/resume_information_extracted.csv",
    index=False
)

# Skill Extractor

In [73]:
skills_df = pd.read_csv("../data/processed/resume_skills.csv")

print(skills_df.shape)
print(skills_df.columns.tolist())

(2481, 8)
['ID', 'resume_text', 'Category', 'clean_resume', 'text_length', 'processed_resume', 'extracted_skills', 'skill_count']


In [74]:
skills_data = skills_df[["ID", "extracted_skills"]].copy()

df = df.merge(
    skills_data,
    on="ID",
    how="left"
)

In [75]:
df = df.rename(
    columns={
        "extracted_skills": "skills"
    }
)

In [76]:
print(df[["ID", "skills"]].head(10).to_string())

         ID                    skills
0  16852973  ['aws', 'data analysis']
1  22323967                   ['git']
2  33176873      ['express', 'excel']
3  27018550                 ['excel']
4  17812897          ['aws', 'excel']
5  11592605                 ['excel']
6  25824789                  ['rest']
7  15375009                   ['aws']
8  11847784          ['aws', 'excel']
9  32896934                   ['aws']


In [77]:
print("Total resumes:", len(df))
print("Resumes with skills:", df["skills"].notna().sum())
print("Resumes without skills:", df["skills"].isna().sum())

Total resumes: 2481
Resumes with skills: 2481
Resumes without skills: 0


In [78]:
df.to_csv(
    "../data/processed/resume_information_extracted.csv",
    index=False
)

In [79]:
print(df.shape)
print(df.columns.tolist())

(2481, 13)
['ID', 'resume_text', 'Category', 'clean_resume', 'text_length', 'email', 'phone', 'name', 'education', 'experience', 'date_experience', 'final_experience', 'skills']
